# Notebook 09: FAISS Vector Search & Dense Retrieval

## KLEAN Chatbot — Restaurant Information Retrieval & Recommendation

### Overview
In this notebook, we evaluate **Dense Vector Search** powered by **FAISS** (Facebook AI Similarity Search) and `VectorSearcher`.

Unlike lexical search (BM25) which requires exact token matches, vector search finds restaurants based on **conceptual meaning**, **synonyms**, and **mood/atmosphere** descriptions.

**Core Objectives:**
1. Load the precomputed FAISS `IndexFlatIP` index (698 restaurants, 384 dimensions).
2. Execute conceptual and natural-language queries (e.g. *"cozy quiet place for romantic date"*, *"cheap local breakfast street food"*).
3. Conduct side-by-side comparison between **BM25** (lexical) and **Dense Vector Search** (semantic).
4. Analyze the **vocabulary mismatch problem** and how dense search solves it.
5. Benchmark retrieval latency.

In [1]:
import sys
import time
from pathlib import Path
import pandas as pd

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.data_loader import DataLoader
from src.utils.document_converter import DocumentConverter
from src.preprocessing.text_preprocessing import TextPreprocessor
from src.indexing.inverted_index import InvertedIndex
from src.retrieval.bm25 import BM25Retriever
from src.embeddings.vector_store import VectorStore
from src.embeddings.embedding_model import EmbeddingModel
from src.retrieval.vector_search import VectorSearcher

print(f"Project root: {PROJECT_ROOT}")

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: D:\Year4\Web_Retrive\Project


## 1. Load Data and Persistent Vector Store

In [2]:
loader = DataLoader(PROJECT_ROOT / "dataset" / "restaurants.json")
records = loader.load()

converter = DocumentConverter()
docs = converter.convert_all(records)

# Load FAISS Vector Store
idx_path = PROJECT_ROOT / "vector_db" / "faiss_index.bin"
meta_path = PROJECT_ROOT / "vector_db" / "index_metadata.json"

vector_store = VectorStore.load(idx_path, meta_path)
print(f"Loaded FAISS Vector Store. Total indexed vectors: {vector_store.total_vectors}")

embedding_model = EmbeddingModel()
vector_searcher = VectorSearcher(vector_store=vector_store, embedding_model=embedding_model)
print("VectorSearcher initialized.")

Loaded FAISS Vector Store. Total indexed vectors: 698


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2693.92it/s]


VectorSearcher initialized.


## 2. Initialize BM25 for Comparison

In [3]:
corpus = converter.get_corpus()
preprocessor = TextPreprocessor()
tokenized_corpus = preprocessor.preprocess_documents(corpus)

index = InvertedIndex()
index.build(tokenized_corpus)
bm25 = BM25Retriever(index, preprocessor)
print(f"BM25 initialized over {len(tokenized_corpus)} documents.")

| 2026-11-22 22:02:29,301 | INFO | khmer-nltk | Loaded model from c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\khmernltk\word_tokenize\sklearn_crf_ner_10000.sav |


BM25 initialized over 698 documents.


## 3. Semantic Queries (Concept & Vibe Search)

Dense retrieval excels at queries describing the atmosphere, vibe, or generic concepts rather than specific brand names.

In [4]:
test_queries = [
    "romantic cozy dinner spot with good wine",
    "cheap local morning soup and noodles",
    "riverside craft beer and sunset view",
    "authentic spicy Sichuan Chinese dining",
]

for q in test_queries:
    print(f"\n{'='*75}\nQUERY: '{q}'\n{'='*75}")
    results = vector_searcher.search(q, top_k=3)
    for r in results:
        rec = converter.get_record(r['doc_id'])
        print(f"Rank {r['rank']} [Sim: {r['score']:.4f}] | {rec['name']}")
        print(f"  Category   : {rec['category']}")
        print(f"  Cuisine    : {', '.join(rec['sub_category']) if rec['sub_category'] else 'N/A'}")
        print(f"  Address    : {rec['address']}")


QUERY: 'romantic cozy dinner spot with good wine'
Rank 1 [Sim: 0.8184] | SWEET HOME restaurant
  Category   : Bar And Grill Restaurant
  Cuisine    : Asian Restaurant, Wine Bar
  Address    : rue 2005, ភ្នំពេញ
Rank 2 [Sim: 0.8073] | ស៊ុបឆ្ងាញ់ 168
  Category   : Soup Restaurant
  Cuisine    : Vietnamese Restaurant, Restaurant
  Address    : Street 99, ភ្នំពេញ
Rank 3 [Sim: 0.8071] | Casa Sach Krok Gourmet
  Category   : Bar And Grill Restaurant
  Cuisine    : N/A
  Address    : Ground floor of The Corner of Bassaclane, ភ្នំពេញ

QUERY: 'cheap local morning soup and noodles'
Rank 1 [Sim: 0.8347] | Seng Heng Noodle Soup
  Category   : Soup Restaurant
  Cuisine    : Restaurant
  Address    : St 298, ភ្នំពេញ
Rank 2 [Sim: 0.8318] | Good Beef Soup
  Category   : Soup Restaurant
  Cuisine    : Restaurant
  Address    : #429ABC, Preah Monivong Blvd (93), ភ្នំពេញ
Rank 3 [Sim: 0.8307] | Mom's Karony
  Category   : Noodles Restaurant
  Cuisine    : Professional Services, Food Delivery Service
  Ad

## 4. Side-by-Side Comparison: BM25 vs. Vector Search

Let us observe queries where **vocabulary mismatch** causes BM25 to return 0 or poor results, while Dense Vector Search retrieves highly relevant places.

In [5]:
comparison_queries = [
    # Query using synonyms not explicitly in the menu
    "wood fired artisan pizza slice",
    # Conceptual vibe
    "quiet cafe for remote laptop working with specialty espresso",
    # Specific dish synonym
    "morning meal with grilled pork and broken rice",
]

for q in comparison_queries:
    print(f"\n{'#'*75}\nQUERY: '{q}'\n{'#'*75}")
    
    bm25_res = bm25.search(q, top_k=3)
    vec_res = vector_searcher.search(q, top_k=3)
    
    print("--- BM25 (Lexical) ---")
    if not bm25_res:
        print("  [No results found]")
    else:
        for r in bm25_res:
            name = converter.get_record(r["doc_id"])["name"]
            print(f"  [{r['rank']}] {name} (BM25 Score: {r['score']:.4f})")
            
    print("\n--- Vector Search (Dense FAISS) ---")
    for r in vec_res:
        name = converter.get_record(r["doc_id"])["name"]
        print(f"  [{r['rank']}] {name} (Cosine Sim: {r['score']:.4f})")


###########################################################################
QUERY: 'wood fired artisan pizza slice'
###########################################################################
--- BM25 (Lexical) ---
  [1] Pocket Pizza - Cambodia (BM25 Score: 5.8101)

--- Vector Search (Dense FAISS) ---
  [1] មីទាញ គាវ សាច់អាំង (Chinese Special Handmade Noodle) (Cosine Sim: 0.7986)
  [2] គាវស្រួយ (Cosine Sim: 0.7883)
  [3] Butterfly Patisserie (Cosine Sim: 0.7874)

###########################################################################
QUERY: 'quiet cafe for remote laptop working with specialty espresso'
###########################################################################
--- BM25 (Lexical) ---
  [1] Cheata Cafe and Restaurant (BM25 Score: 5.1388)
  [2] Chamkar Nonong Restuarant & Cafe (BM25 Score: 4.7907)
  [3] K.E Cafe (BM25 Score: 4.7907)

--- Vector Search (Dense FAISS) ---
  [1] Re Kuyteav (Cosine Sim: 0.8207)
  [2] Bella.Tek (Cosine Sim: 0.8148)
  [3] Tube Coffee (Cosin

## 5. Latency Benchmark

Dense query encoding + FAISS inner product search is benchmarked over 50 iterations.

In [6]:
latencies = []
for _ in range(50):
    t0 = time.perf_counter()
    _ = vector_searcher.search("spicy khmer noodle soup", top_k=5)
    latencies.append((time.perf_counter() - t0) * 1000) # milliseconds

print(f"Vector Search Latency (Query Encoding + FAISS Search):")
print(f"  Mean Latency : {sum(latencies)/len(latencies):.2f} ms")
print(f"  Min Latency  : {min(latencies):.2f} ms")
print(f"  Max Latency  : {max(latencies):.2f} ms")

Vector Search Latency (Query Encoding + FAISS Search):
  Mean Latency : 16.73 ms
  Min Latency  : 12.06 ms
  Max Latency  : 23.48 ms


## 6. Findings & Path to Hybrid Search (Phase 5)

| Aspect | Lexical Search (BM25) | Dense Vector Search (FAISS) |
|---|---|---|
| **Matching Mechanism** | Exact word & sub-word tokens | Dense continuous embedding vectors |
| **Synonyms & Concepts** | Fails unless words appear verbatim | Exceptional semantic generalization |
| **Exact Identifiers (Street No., Phone, Exact Name)** | Highly precise ($O(1)$ token match) | Can hallucinate or drift to semantically nearby items |
| **Multilingual Support** | Requires explicit dictionary mapping | Cross-lingual alignment built-in |

### Why Hybrid Retrieval is Needed (Phase 5):
Neither BM25 nor Vector Search alone is optimal for real-world user queries.
- Users searching for *"Restaurant 126"* need **BM25's exact token precision**.
- Users searching for *"chill dinner date place with cocktails"* need **Vector Search's semantic understanding**.

In **Phase 5**, we combine both using **Reciprocal Rank Fusion (RRF)** to deliver state-of-the-art hybrid retrieval.